In [ ]:
from spiral import Spiral

sp = Spiral(overrides={
    "keys_cache.enabled": "1",
    "keys_cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    "keys_cache.disk_capacity_bytes": "0",
    # "fragments_cache.enabled": "1",
    # "fragments_cache.memory_capacity_bytes": "4294967296",  # 4 GiB
    # "fragments_cache.memory_capacity_bytes": "0",
    "fragments_cache.disk_capacity_bytes": "0",
    "manifests_cache.enabled": "1",
    "manifests_cache.memory_capacity_bytes": "0",
    "manifests_cache.disk_capacity_bytes": "1073741824",  # 1GiB
})

In [ ]:
project = sp.project("enigma-spiral-poc-2-724186")

In [ ]:
project.list_tables()

In [ ]:
tbl_session_reconstructed_video_metadata = project.table("session_reconstructed_video_metadata")
tbl_spike_data_by_time = project.table("spike_data_by_time")
tbl_vidtok_embeddings = project.table("vidtok_embeddings")
tbl_vjepa_embeddings = project.table("vjepa_embeddings")
tbl_behavior_adc = project.table("behavior_adc")
tbl_stim_events = project.table("stim_events")

In [ ]:
# Random explorations.
tbl_behavior_adc.schema()

In [ ]:
# Random explorations.
tbl_session_reconstructed_video_metadata.to_polars_lazy_frame().head().collect()

In [ ]:
import pyarrow as pa
from spiral import Shard, KeyRange

def ranges_to_shards(tbl, list_ranges) -> list[Shard]:
    shards = []
    for r in list_ranges:
        st = tbl.key(r["session_id"], r["start"])
        ed = tbl.key(r["session_id"], r["end"])
        key_range = KeyRange(begin=st, end=ed)
        shards.append(Shard(key_range, None))
    return shards

time_ranges = sp.scan({
    "session_id": tbl_session_reconstructed_video_metadata["session_id"],
    "start": tbl_session_reconstructed_video_metadata["timestamp"],
    "end": tbl_session_reconstructed_video_metadata["timestamp"] + 1_000_000,
}).to_table().to_pylist()
sessions_shards = ranges_to_shards(tbl_session_reconstructed_video_metadata, time_ranges)

len(sessions_shards)

In [ ]:
from spiral import Sampler

def behavior_sampler_function(array: pa.Array) -> pa.Array:
    return pa.array([i % 10 == 0 for i in range(len(array))])

behavior_sampler = Sampler(behavior_sampler_function)

In [ ]:
import numpy as np

def stack_item(item):
    """
    Convert a single batch item to numpy arrays (most efficient version).
    Handles variable lengths: behavior can be 600 or 601, vidtok can have 7 or 8 embeddings.

    Args:
        item: Tuple of (behavior_batch, embeddings_batch)

    Returns:
        dict[str, np.ndarray]: Dictionary with 'behavior' and 'vidtok' arrays
    """
    behavior_batch = item[0]
    embeddings_batch = item[1]
    vjepa_batch = item[2]

    # Stack behavior data (600 or 601, 3)
    behavior_array = np.column_stack([
        behavior_batch["pupil_size_in"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_x_px_offset_center"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_y_px_offset_center"].to_numpy(zero_copy_only=False)
    ])

    # Stack vidtok embeddings
    tensor_column = embeddings_batch["tensor"]
    num_vidtoks = len(tensor_column)

    # Double flatten: list<list<float>> -> flat float array
    flattened_once = pa.ListArray.flatten(tensor_column)
    flattened_twice = pa.ListArray.flatten(flattened_once)

    # Zero-copy to numpy
    vidtok_flat = flattened_twice.to_numpy(zero_copy_only=False)

    # Infer number of rows per vidtok (should be 1590)
    num_vidtok_rows = len(vidtok_flat) // (num_vidtoks * 16)

    # Reshape and transpose: (num_vidtoks, num_vidtok_rows, 16) -> (num_vidtok_rows, 16, num_vidtoks)
    vidtok_array = vidtok_flat.reshape(num_vidtoks, num_vidtok_rows, 16).transpose(1, 2, 0)

    # Stack vjepa embeddings (15 rows of (33792, 24) -> (1408, 24, 24, 15))
    vjepa_tensor_column = vjepa_batch["tensor"]
    num_vjepa = len(vjepa_tensor_column)  # Should be 15

    # Double flatten: list<list<float>> -> flat float array
    vjepa_flattened_once = pa.ListArray.flatten(vjepa_tensor_column)
    vjepa_flattened_twice = pa.ListArray.flatten(vjepa_flattened_once)

    # Zero-copy to numpy
    vjepa_flat = vjepa_flattened_twice.to_numpy(zero_copy_only=False)

    # Reshape: (15 * 33792 * 24) -> (15, 33792, 24) -> (15, 1408, 24, 24) -> (1408, 24, 24, 15)
    vjepa_array = vjepa_flat.reshape(num_vjepa, 33792, 24).reshape(num_vjepa, 1408, 24, 24).transpose(1, 2, 3, 0)

    return {
        "behavior": behavior_array,  # Shape: (600 or 601, 3)
        "vidtok": vidtok_array,      # Shape: (1590, 16, 7 or 8)
        "vjepa": vjepa_array         # Shape: (1408, 24, 24, 15)
    }

In [ ]:
# Open scans once.
vidtok_embeddings_scan = sp.scan(tbl_vidtok_embeddings["tensor"], where=tbl_vidtok_embeddings["modality"] == "rgb")
vjepa_embeddings_scan = sp.scan(tbl_vjepa_embeddings["tensor"], where=tbl_vjepa_embeddings["layer"] == 4)

In [ ]:
import tqdm

vjepa_embeddings_loader = vjepa_embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=32)
for item in tqdm.tqdm(vjepa_embeddings_loader):
    pass

In [ ]:
import tqdm
import pyarrow as pa

# Must go through sp.sample to apply the sampler
behavior_loader: pa.RecordBatchReader = sp.sample(
    tbl_behavior_adc[["pupil_size_in", "eye_x_px_offset_center", "eye_y_px_offset_center"]],
    sampler=behavior_sampler,
    shards=sessions_shards,
    batch_readahead=64
)

vidtok_embeddings_loader = vidtok_embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=32)
vjepa_embeddings_loader = vjepa_embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=32)

for item in tqdm.tqdm(zip(behavior_loader, vidtok_embeddings_loader, vjepa_embeddings_loader)):
    sample = stack_item(item)

    # print({
    #     key: value.shape for key, value in sample.items()
    # })
    # break